### Data Preprocessing

This notebook shows the pipeline of data preprocessing

### Initialize

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import os
import h5py
import math
from scipy.signal import stft
from typing import List, Tuple
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEBUG = False

### Load Data

In [4]:
# MultiFileDataset(Dataset)
class MultiFileDataset(Dataset):
    def __init__(self, file_paths, feature_dim):
        self.file_paths = file_paths
        self.index_map = []  

        for file_idx, file_path in enumerate(file_paths):  
            with h5py.File(file_path, 'r') as f:
                num_samples = len(f.keys())  
                self.index_map.extend([(file_idx, i) for i in range(num_samples)])

        self.data = {}

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        file_idx, sample_idx = self.index_map[idx]   
        file_path = self.file_paths[file_idx]

        with h5py.File(file_path, 'r') as f:
            sample_key = list(f.keys())[sample_idx]
            g = f[sample_key]

            self.data['neural_features'] = torch.tensor(g['input_features'][:,feature_dim[0]:feature_dim[1]])#[feature_dim[0]:feature_dim[1]]
            self.data['n_time_steps'] = torch.tensor(g.attrs['n_time_steps'])
            self.data['seq_class_ids'] = torch.tensor(g['seq_class_ids'][:] if 'seq_class_ids' in g else None)
            # self.data['pred_seq'] = [LOGIT_TO_PHONEME[p] for p in self.data['seq_class_ids']]
            self.data['seq_len'] = torch.tensor(g.attrs['seq_len'] if 'seq_len' in g.attrs else None)
            # self.data['transcriptions'] = g['transcription'][:] if 'transcription' in g else None
            self.data['sentence_label'] = g.attrs['sentence_label'][:] if 'sentence_label' in g.attrs else None
            # self.data['session'] = g.attrs['session']
            # self.data['block_num'] = g.attrs['block_num']
            # self.data['trial_num'] = g.attrs['trial_num']

        return self.data['neural_features'], self.data['n_time_steps'], self.data['seq_class_ids'][:self.data['seq_len']]#, self.data['sentence_label']

In [ ]:
# check the settings of original hdf5 files
data_dir = "../data/hdf5_data_final"
folders = [os.path.join(data_dir, folder) for folder in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, folder))]

files = []
for folder in folders:
    files.extend([os.path.join(folder, file) for file in os.listdir(folder) if file == "data_train.hdf5"])

with h5py.File(files[0], "r") as f:
    def print_compression(name, obj):
        if isinstance(obj, h5py.Dataset): 
            print(f"Dataset: {name}")
            print(f"  Compression: {obj.compression}")  
            print(f"  Compression options: {obj.compression_opts}")  
            print(f"  Shape: {obj.shape}")
            print(f"  Data type: {obj.dtype}")
            print()

    f.visititems(print_compression)

Dataset: trial_0000/input_features
  Compression: lzf
  Compression options: None
  Shape: (321, 512)
  Data type: float32

Dataset: trial_0000/seq_class_ids
  Compression: lzf
  Compression options: None
  Shape: (500,)
  Data type: int32

Dataset: trial_0000/transcription
  Compression: lzf
  Compression options: None
  Shape: (500,)
  Data type: int32

Dataset: trial_0001/input_features
  Compression: lzf
  Compression options: None
  Shape: (481, 512)
  Data type: float32

Dataset: trial_0001/seq_class_ids
  Compression: lzf
  Compression options: None
  Shape: (500,)
  Data type: int32

Dataset: trial_0001/transcription
  Compression: lzf
  Compression options: None
  Shape: (500,)
  Data type: int32

Dataset: trial_0002/input_features
  Compression: lzf
  Compression options: None
  Shape: (480, 512)
  Data type: float32

Dataset: trial_0002/seq_class_ids
  Compression: lzf
  Compression options: None
  Shape: (500,)
  Data type: int32

Dataset: trial_0002/transcription
  Compres

### Process and Save

In [ ]:
# Do STFT, then stack the first stack_layers frequency bands to all samples and save to new HDF5 files

split = 'test'    # which split to process: 'train', 'val', 'test'

def process_features(x_np, fs=50, nperseg=32, noverlap=28, stack_layers=17):   # x_np is (T, F)
    # x_np: (T, F) -> (F, T), STFT feature by feature
    x = x_np.T
    C, T = x.shape
    stft_list = []
    for ch in range(C):
        f, t, Zxx = stft(x[ch], fs=fs, nperseg=nperseg, noverlap=noverlap)
        stft_list.append(np.abs(Zxx))  # (freq_bins, time_bins)
    stft_array = np.stack(stft_list, axis=0)  # all layers
    stacked = np.stack(stft_array[:, 0:stack_layers, :], axis=0)  # the first stack_layers layer, (Freq, stack, T_stft)
    stft_stack = torch.from_numpy(stacked).float().permute(1, 0, 2)  # (stack, Freq, T_stft)
    return stft_stack.numpy().astype(np.float32)  # save as float32

def save_all(data_root="../data/hdf5_data_final", out_root="../data/processed_hdf5"):   # modify paths as needed
    os.makedirs(out_root, exist_ok=True)
    sessions = [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))]

    for sess in sessions:
        in_file = os.path.join(data_root, sess, f"data_{split}.hdf5")
        if not os.path.exists(in_file):
            continue
        out_dir = os.path.join(out_root, sess)
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, f"data_{split}_processed.hdf5")
        if os.path.exists(out_file):
            print(f"Skip existing {out_file}")
            continue

        with h5py.File(in_file, "r") as fin, h5py.File(out_file, "w") as fout:
            keys = list(fin.keys())
            for k in keys:
                g_in = fin[k]
                x = g_in["input_features"][:]         # (T, F)
                x_proc = process_features(x)          # (C_proc, Freq, T_stft) or shape you want
                grp = fout.create_group(k)
                # Use chunk + compression, chunk along time dimension
                # print(g_in['input_features'][:])
                grp.create_dataset("input_features", data=g_in['input_features'], dtype="float32", compression="lzf")
                grp.create_dataset(
                    "time_freq_features",
                    data=torch.tensor(x_proc),
                    dtype="float32",
                    compression="lzf",
                    chunks=(min(x_proc.shape[0], 4), x_proc.shape[1], min(x_proc.shape[2], 128)),
                )
                if 'seq_class_ids' in g_in:
                    grp.create_dataset("seq_class_ids", data=g_in["seq_class_ids"], dtype="int32", compression="lzf") 
                if 'seq_len' in g_in.attrs:
                    grp.attrs["seq_len"] = g_in.attrs["seq_len"]
                if 'sentence_label' in g_in.attrs:
                    grp.attrs['sentence_label'] = g_in.attrs['sentence_label'][:]
                
                grp.attrs["n_time_steps"] = g_in.attrs["n_time_steps"]
                grp.attrs['session'] = g_in.attrs['session']
                grp.attrs['block_num'] = g_in.attrs['block_num']
                grp.attrs['trial_num'] = g_in.attrs['trial_num']

        print(f"Saved {out_file}")
    
# save_all()    # uncomment to save. current settings will take roughly 100 GB.